# Autonomous Code Writer

**Multi-Agent GitHub Workflow** | LangGraph + LangChain | AI Agents Course Project

---

## What it does

You give it a GitHub repo URL. It clones the repo, analyzes it, suggests a useful feature, **asks you** if the idea is good, creates a GitHub issue + branch, implements the change, runs tests, **asks you again** if the result looks right, opens a pull request, and reviews its own diff — leaving the final merge decision to you.

## Two human approval gates

| Gate | When | Your options |
|------|------|--------------|
| **1** | After feature proposal | `approve` → create issue & branch, `reject` → stop, `revise` → regenerate proposal |
| **2** | After implementation | `approve` → push branch & open PR, `reject` → stop, `revise` → redo implementation |

## Five agents

1. **Repository Analyst** — clones & inspects the repo (language, framework, tests)
2. **Feature Strategist** — proposes a conservative, useful feature
3. **GitHub Coordinator** — creates issues, branches, PRs, comments
4. **Implementation Agent** — makes focused code changes on the branch
5. **PR Review Agent** — self-reviews the diff and posts a GitHub comment

## Safety rules

- Never pushes to the default branch
- Never merges pull requests
- Never creates GitHub artifacts without your approval
- API keys loaded from env / Colab Secrets only

## Architecture

```
Request → Clone/Analyze → Propose → [HITL 1] → Issue+Branch → Implement → Verify → [HITL 2] → PR → Review → Done
```

## Assignment checklist

| Requirement | Status |
|-------------|--------|
| ≥2 agents | 5 agents |
| LangGraph stateful workflow | StateGraph + checkpointing |
| LangChain model/tools | OpenAI + tool wrappers |
| Multiple tools | Clone, inspect, GitHub API, test/lint |
| Conversational memory | Memory checkpointer |
| Human-in-the-Loop | 2 approval gates with revise loops |
| `execute_workflow(request)` | Public function |
| 5 test cases | 5 distinct calls on 1 demo repo |
| Approval test | Happy path |
| Revision test | Feature + implementation revision |

---

**Full details:** [README.md](README.md) — architecture diagram, state schema, agent breakdown, requirements mapping, safety docs, Colab setup guide.

## 1. Runtime Setup — Dependencies, Secrets, Model Init (Issue 2)

This section installs required packages, loads credentials securely, and initializes the OpenAI chat model and GitHub client.

### Required GitHub Token Permissions

Your `GITHUB_TOKEN` needs the following permissions for the target repository:

- **Contents** — read & write (clone, create branches, push commits)
- **Issues** — read & write (create issues)
- **Pull requests** — read & write (create PRs, post review comments)
- **Metadata** — read-only (read repo info, default branch)

If using a **classic token**, ensure the `repo` scope is enabled.
If using a **fine-grained token**, select the four permissions above for the specific repository.

### Secret Loading Rules

- `OPENAI_API_KEY` and `GITHUB_TOKEN` are loaded from environment variables or Google Colab Secrets **only**.
- Hardcoding secrets in the notebook is prohibited.
- Missing secrets produce clear errors without exposing secret values.
- Secrets are never printed to notebook output.

In [ ]:
!pip install -q langchain langgraph langchain-openai openai pygithub gitpython

In [ ]:
!pip install -q python-dotenv
from dotenv import load_dotenv

# Load .env file
load_dotenv()

In [ ]:
import os
import sys

# Attempt to load secrets from Google Colab Secrets (preferred in Colab environment)
try:
    from google.colab import userdata
    os.environ.setdefault('OPENAI_API_KEY', userdata.get('OPENAI_API_KEY') or '')
    os.environ.setdefault('GITHUB_TOKEN', userdata.get('GITHUB_TOKEN') or '')
except ImportError:
    pass  # Not running in Colab; rely on environment variables

OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '').strip()
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '').strip()

# Validate secrets without exposing values
missing = []
if not OPENAI_API_KEY:
    missing.append('OPENAI_API_KEY')
if not GITHUB_TOKEN:
    missing.append('GITHUB_TOKEN')

if missing:
    print('ERROR: Missing required secrets. Please set them via environment variables or Colab Secrets.')
    for m in missing:
        print(f'  - {m} is not set')
    print('Setup cannot continue until the secrets are configured.')
    sys.exit(1)
else:
    print('Secrets loaded successfully. (Values are hidden for security.)')

In [ ]:
from langchain_openai import ChatOpenAI

DEFAULT_MODEL_NAME = os.environ.get('OPENAI_MODEL_NAME', 'gpt-5.4-mini-2026-03-17')

llm = ChatOpenAI(
    model=DEFAULT_MODEL_NAME,
    temperature=0.2,
    openai_api_key=OPENAI_API_KEY,
)

print(f'OpenAI model initialized: {DEFAULT_MODEL_NAME}')

In [ ]:
from github import Github

github_client = Github(GITHUB_TOKEN, timeout=30)

try:
    user = github_client.get_user()
    login = user.login
    print(f'GitHub client authenticated as: {login}')
except Exception as e:
    print(f'ERROR: GitHub authentication failed — {e}')
    sys.exit(1)

# Verify repository access with a lightweight metadata call (optional sanity check)
# Replace with your demo repo if you want to test access immediately:
repo = github_client.get_repo('BozhidarN7/ds-algo-visualizer')
print(f'Repo access OK: {repo.full_name}')

## Implementation (Issues 3–12)

The sections below will be populated as the remaining issues are completed.

2. **Repository Analysis** — clone, inspect, detect stack (Issue 3)
3. **Feature Strategy & HITL Gate 1** — generate proposal, interrupt (Issue 4)
4. **GitHub Coordination** — create issue and branch (Issue 5)
5. **Implementation** — plan and execute changes (Issue 6)
6. **Verification & PR Prep** — run tests/lint, build PR metadata (Issue 7)
7. **HITL Gate 2 & PR Creation** — approve/revise, push, open PR (Issue 8)
8. **PR Review & Comment** — self-review diff, post comment (Issue 9)
9. **`execute_workflow` Interface** — end-to-end function with memory (Issue 10)
10. **Test Cases** — five reproducible executions (Issue 11)
11. **Final Docs** — safety, limitations, submission (Issue 12)

In [ ]:
# Issue 2 complete: Runtime setup ready.
print('Issue 2 complete: Dependencies installed, secrets loaded, OpenAI model and GitHub client initialized.')